In [1]:
# ============================================================
# CELL 0 — INSTALL
# ============================================================

import subprocess
import sys

def pip(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True
    )

pip("ultralytics==8.3.*")

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.4 MB/s eta 0:00:00
Installation complete.


In [2]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

from pathlib import Path
import random
import shutil
import cv2
import numpy as np
import torch
import yaml
from ultralytics import YOLO

# Existing trained model
MODEL_PATH = Path(
    "/kaggle/input/notebooks/tanvirmahtabtaneem/"
    "ocr-model/bangla_lpdb_runs/"
    "bangla_lpdb_yolo11m_t4x2/"
    "weights/best.pt"
)

# Public Bangla LPDB-A dataset
DATASET_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/"
    "ocr-dataset/Bangla LPDB - A/Bangla_License_Plate"
)

CLASS_FILE = DATASET_ROOT / "classes.txt"

# New fine-tuning dataset
WORK_ROOT = Path(
    "/kaggle/working/bangla_lpdb_finetune"
)

RUN_ROOT = Path(
    "/kaggle/working/bangla_lpdb_finetune_runs"
)

DATA_YAML = WORK_ROOT / "data.yaml"

SEED = 42

# We deliberately use ONE T4 here to avoid the notebook DDP error.
DEVICE = 0

IMG_SIZE = 960
BATCH_SIZE = 4
EPOCHS = 35
WORKERS = 2

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# Number of synthetic CCTV-degraded copies
# for each clean training image.
DEGRADED_COPIES = 2

print("Configuration loaded.")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Configuration loaded.


In [3]:
# ============================================================
# CELL 2 — REPRODUCIBILITY + GPU CHECK
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable GPU in Kaggle."
    )

print("This fine-tuning run uses GPU 0 only.")


GPU CHECK
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
This fine-tuning run uses GPU 0 only.


In [4]:
# ============================================================
# CELL 3 — CHECK MODEL AND DATASET PATHS
# ============================================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Existing best.pt not found:\n{MODEL_PATH}"
    )

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found:\n{DATASET_ROOT}"
    )

if not CLASS_FILE.exists():
    raise FileNotFoundError(
        f"classes.txt not found:\n{CLASS_FILE}"
    )

print("Existing model found:")
print(MODEL_PATH)

print("\nDataset found:")
print(DATASET_ROOT)



Existing model found:
/kaggle/input/notebooks/tanvirmahtabtaneem/ocr-model/bangla_lpdb_runs/bangla_lpdb_yolo11m_t4x2/weights/best.pt

Dataset found:
/kaggle/input/datasets/tanvirmahtabtaneem/ocr-dataset/Bangla LPDB - A/Bangla_License_Plate


In [5]:
# ============================================================
# CELL 4 — LOAD 102 CLASSES
# ============================================================

class_names = [
    x.strip()
    for x in CLASS_FILE.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()
    if x.strip()
]

print("Number of classes:", len(class_names))

if len(class_names) != 102:
    raise RuntimeError(
        f"Expected 102 classes, found {len(class_names)}."
    )

for i, name in enumerate(class_names):
    print(f"{i:3d} -> {name}")

VALID_CLASS_IDS = set(
    range(len(class_names))
)



Number of classes: 102
  0 -> 0
  1 -> 1
  2 -> 2
  3 -> 3
  4 -> 4
  5 -> 5
  6 -> 6
  7 -> 7
  8 -> 8
  9 -> 9
 10 -> Metro
 11 -> A
 12 -> Bha
 13 -> Cha
 14 -> Chha
 15 -> Da
 16 -> DA
 17 -> E
 18 -> Ga
 19 -> Gha
 20 -> Ha
 21 -> Ja
 22 -> Jha
 23 -> Ka
 24 -> Kha
 25 -> La
 26 -> Ma
 27 -> Na
 28 -> Pa
 29 -> Sa
 30 -> Sha
 31 -> Ta
 32 -> THA
 33 -> Tha
 34 -> U
 35 -> Bagerhat
 36 -> Bagura
 37 -> Bandarban
 38 -> Barguna
 39 -> Barisal
 40 -> Bhola
 41 -> Brahmanbaria
 42 -> Chandpur
 43 -> Chapainawabganj
 44 -> Chatto
 45 -> Chattogram
 46 -> Chuadanga
 47 -> Cox's Bazar
 48 -> Cumilla
 49 -> Dhaka
 50 -> Dinajpur
 51 -> Faridpur
 52 -> Feni
 53 -> Gaibandha
 54 -> Gazipur
 55 -> Gopalganj
 56 -> Habiganj
 57 -> Jamalpur
 58 -> Jessore
 59 -> Jhalokati
 60 -> Jhenaidah
 61 -> Joypurhat
 62 -> Khagrachari
 63 -> Khulna
 64 -> Kishoreganj
 65 -> Kurigram
 66 -> Kustia
 67 -> Lakshmipur
 68 -> Lalmonirhat
 69 -> Madaripur
 70 -> Magura
 71 -> Manikganj
 72 -> Meherpur
 73 -> M

In [6]:
# ============================================================
# CELL 5 — FIND IMAGE/LABEL PAIRS
# ============================================================

images = sorted([
    p
    for p in DATASET_ROOT.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

pairs = []

for image_path in images:
    label_path = image_path.with_suffix(".txt")

    if label_path.exists():
        pairs.append(
            (image_path, label_path)
        )

print("Images:", len(images))
print("Image/label pairs:", len(pairs))

if len(pairs) != 2668:
    print(
        "WARNING: Expected about 2668 pairs for LPDB-A."
    )

if not pairs:
    raise RuntimeError(
        "No image/label pairs were found."
    )



Images: 2668
Image/label pairs: 2668


In [7]:
# ============================================================
# CELL 6 — CHECK LABEL IDS
# ============================================================

all_class_ids = []

for _, label_path in pairs:

    for line in label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        try:
            class_id = int(float(parts[0]))
            all_class_ids.append(class_id)
        except ValueError:
            pass

print(
    "Minimum class ID:",
    min(all_class_ids)
)

print(
    "Maximum class ID:",
    max(all_class_ids)
)

extra_ids = sorted(
    x
    for x in set(all_class_ids)
    if x not in VALID_CLASS_IDS
)

print(
    "Invalid IDs to skip:",
    extra_ids
)

Minimum class ID: 0
Maximum class ID: 105
Invalid IDs to skip: [102, 103, 105]


In [8]:
# ============================================================
# CELL 7 — NORMALIZE YOLO LABEL FILES
# ============================================================

def normalize_label_file(
    source_path,
    destination_path
):

    output = []

    lines = source_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    for line in lines:

        parts = line.split()

        if len(parts) != 5:
            continue

        try:
            class_id = int(float(parts[0]))

            x = float(parts[1])
            y = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

        except ValueError:
            continue

        # Dataset is already 0-based.
        # Keep only classes 0..101.
        if class_id not in VALID_CLASS_IDS:
            continue

        if not all(
            0.0 <= value <= 1.0
            for value in [x, y, w, h]
        ):
            continue

        output.append(
            f"{class_id} "
            f"{x:.6f} "
            f"{y:.6f} "
            f"{w:.6f} "
            f"{h:.6f}"
        )

    destination_path.write_text(
        "\n".join(output),
        encoding="utf-8"
    )

print("Label normalization function ready.")



Label normalization function ready.


In [9]:
# ============================================================
# CELL 8 — TRAIN / VAL / TEST SPLIT
# ============================================================

random.shuffle(pairs)

total = len(pairs)

train_end = int(
    total * TRAIN_RATIO
)

val_end = (
    train_end
    + int(total * VAL_RATIO)
)

train_pairs = pairs[:train_end]
val_pairs = pairs[train_end:val_end]
test_pairs = pairs[val_end:]

print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))

if not val_pairs:
    raise RuntimeError("Validation split is empty.")

if not test_pairs:
    raise RuntimeError("Test split is empty.")


Train: 2134
Val  : 266
Test : 268


In [10]:
# ============================================================
# CELL 9 — CCTV DEGRADATION FUNCTIONS
# ============================================================

def horizontal_motion_blur(
    image,
    kernel_size
):

    kernel = np.zeros(
        (kernel_size, kernel_size),
        dtype=np.float32
    )

    center = kernel_size // 2

    kernel[center, :] = (
        1.0 / kernel_size
    )

    return cv2.filter2D(
        image,
        -1,
        kernel
    )


def jpeg_compress(
    image,
    quality
):

    ok, encoded = cv2.imencode(
        ".jpg",
        image,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            quality
        ]
    )

    if not ok:
        return image

    return cv2.imdecode(
        encoded,
        cv2.IMREAD_COLOR
    )


def degrade_plate(image):

    output = image.copy()

    h, w = output.shape[:2]

    # Motion blur
    if random.random() < 0.90:

        kernel_size = random.choice([
            5, 7, 9, 11,
            13, 15, 17,
            19, 21
        ])

        output = horizontal_motion_blur(
            output,
            kernel_size
        )

    # Low resolution
    if random.random() < 0.85:

        scale = random.uniform(
            0.20,
            0.60
        )

        small_w = max(
            32,
            int(w * scale)
        )

        small_h = max(
            16,
            int(h * scale)
        )

        small = cv2.resize(
            output,
            (small_w, small_h),
            interpolation=cv2.INTER_AREA
        )

        output = cv2.resize(
            small,
            (w, h),
            interpolation=cv2.INTER_CUBIC
        )

    # Brightness / contrast
    if random.random() < 0.75:

        alpha = random.uniform(
            0.65,
            1.25
        )

        beta = random.randint(
            -35,
            35
        )

        output = cv2.convertScaleAbs(
            output,
            alpha=alpha,
            beta=beta
        )

    # Noise
    if random.random() < 0.60:

        sigma = random.uniform(
            4.0,
            14.0
        )

        noise = np.random.normal(
            0,
            sigma,
            output.shape
        )

        output = np.clip(
            output.astype(np.float32)
            + noise,
            0,
            255
        ).astype(np.uint8)

    # JPEG compression
    if random.random() < 0.75:

        output = jpeg_compress(
            output,
            random.randint(25, 65)
        )

    # Gamma
    if random.random() < 0.40:

        gamma = random.uniform(
            0.65,
            1.35
        )

        table = np.array([
            ((i / 255.0) ** gamma) * 255
            for i in range(256)
        ]).astype(np.uint8)

        output = cv2.LUT(
            output,
            table
        )

    return output



In [11]:
# ============================================================
# CELL 10 — CREATE FRESH WORKING DATASET
# ============================================================

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

for split in ["train", "val", "test"]:

    (
        WORK_ROOT
        / "images"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        WORK_ROOT
        / "labels"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

print("Working dataset folders created.")



Working dataset folders created.


In [12]:
# ============================================================
# CELL 11 — COPY IMAGE + LABEL FUNCTION
# ============================================================

def copy_pair(
    image_path,
    label_path,
    split,
    suffix=""
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        print(
            "WARNING: could not read:",
            image_path.name
        )
        return False

    stem = (
        image_path.stem
        + suffix
    )

    output_image = (
        WORK_ROOT
        / "images"
        / split
        / f"{stem}.jpg"
    )

    output_label = (
        WORK_ROOT
        / "labels"
        / split
        / f"{stem}.txt"
    )

    saved = cv2.imwrite(
        str(output_image),
        image,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            95
        ]
    )

    if not saved:
        return False

    normalize_label_file(
        label_path,
        output_label
    )

    return True



In [13]:
# ============================================================
# CELL 12 — BUILD TRAINING DATA
# ============================================================

print("Building training data...")

train_clean = 0
train_degraded = 0

for image_path, label_path in train_pairs:

    ok = copy_pair(
        image_path,
        label_path,
        "train"
    )

    if not ok:
        continue

    train_clean += 1

    image = cv2.imread(
        str(image_path)
    )

    for index in range(
        DEGRADED_COPIES
    ):

        degraded = degrade_plate(
            image
        )

        suffix = (
            f"_blur{index + 1}"
        )

        output_image = (
            WORK_ROOT
            / "images"
            / "train"
            / f"{image_path.stem}{suffix}.jpg"
        )

        output_label = (
            WORK_ROOT
            / "labels"
            / "train"
            / f"{image_path.stem}{suffix}.txt"
        )

        cv2.imwrite(
            str(output_image),
            degraded,
            [
                cv2.IMWRITE_JPEG_QUALITY,
                95
            ]
        )

        normalize_label_file(
            label_path,
            output_label
        )

        train_degraded += 1

print("Clean train images:", train_clean)
print("Degraded train images:", train_degraded)




Building training data...


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


Clean train images: 2134
Degraded train images: 4268


In [14]:
# ============================================================
# CELL 13 — BUILD VALIDATION DATA
# ============================================================
#
# THIS CELL WAS MISSING IN THE PREVIOUS NOTEBOOK.
# That is why:
# /kaggle/working/bangla_lpdb_finetune/images/val
# was empty.
# ============================================================

print("Building validation data...")

val_count = 0

for image_path, label_path in val_pairs:

    ok = copy_pair(
        image_path,
        label_path,
        "val"
    )

    if ok:
        val_count += 1

print(
    "Validation images created:",
    val_count
)

if val_count == 0:
    raise RuntimeError(
        "Validation dataset is still empty."
    )



Building validation data...
Validation images created: 266


In [15]:
# ============================================================
# CELL 14 — BUILD TEST DATA
# ============================================================

print("Building test data...")

test_count = 0

for image_path, label_path in test_pairs:

    ok = copy_pair(
        image_path,
        label_path,
        "test"
    )

    if ok:
        test_count += 1

print(
    "Test images created:",
    test_count
)

if test_count == 0:
    raise RuntimeError(
        "Test dataset is empty."
    )

Building test data...
Test images created: 268


In [16]:
# ============================================================
# CELL 15 — CREATE data.yaml + VERIFY DATASET
# ============================================================

yaml_data = {
    "path": str(WORK_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(class_names),
    "names": class_names
}

DATA_YAML.write_text(
    yaml.safe_dump(
        yaml_data,
        allow_unicode=True,
        sort_keys=False
    ),
    encoding="utf-8"
)

train_images = list(
    (WORK_ROOT / "images" / "train").glob("*.jpg")
)

val_images = list(
    (WORK_ROOT / "images" / "val").glob("*.jpg")
)

test_images = list(
    (WORK_ROOT / "images" / "test").glob("*.jpg")
)

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print("Train images:", len(train_images))
print("Val images  :", len(val_images))
print("Test images :", len(test_images))

print(
    "data.yaml:",
    DATA_YAML
)

if len(train_images) == 0:
    raise RuntimeError(
        "No training images found."
    )

if len(val_images) == 0:
    raise RuntimeError(
        "No validation images found."
    )

if len(test_images) == 0:
    raise RuntimeError(
        "No test images found."
    )

# Show some validation files.
print("First validation images:")

for p in val_images[:5]:
    print(p)

print("Dataset is ready.")



DATASET CHECK
Train images: 6402
Val images  : 266
Test images : 268
data.yaml: /kaggle/working/bangla_lpdb_finetune/data.yaml
First validation images:
/kaggle/working/bangla_lpdb_finetune/images/val/2300.jpg
/kaggle/working/bangla_lpdb_finetune/images/val/2606.jpg
/kaggle/working/bangla_lpdb_finetune/images/val/2657.jpg
/kaggle/working/bangla_lpdb_finetune/images/val/2207.jpg
/kaggle/working/bangla_lpdb_finetune/images/val/886.jpg
Dataset is ready.


In [17]:
# ============================================================
# CELL 16 — LOAD EXISTING BEST MODEL
# ============================================================

print("=" * 70)
print("LOADING EXISTING BEST MODEL")
print("=" * 70)

model = YOLO(
    str(MODEL_PATH)
)

print(
    "Loaded:",
    MODEL_PATH
)

LOADING EXISTING BEST MODEL
Loaded: /kaggle/input/notebooks/tanvirmahtabtaneem/ocr-model/bangla_lpdb_runs/bangla_lpdb_yolo11m_t4x2/weights/best.pt


In [18]:
# ============================================================
# CELL 17 — FINE-TUNE ON ONE T4
# ============================================================

print("=" * 70)
print("STARTING FINE-TUNING")
print("=" * 70)

print("Starting model:", MODEL_PATH)
print("Dataset:", DATA_YAML)
print("GPU: 0")
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

results = model.train(

    data=str(DATA_YAML),

    epochs=EPOCHS,

    imgsz=IMG_SIZE,

    batch=BATCH_SIZE,

    # ONE T4 — avoids the previous notebook DDP problem.
    device=DEVICE,

    workers=WORKERS,

    optimizer="AdamW",

    # Lower LR for fine-tuning.
    lr0=0.0002,

    lrf=0.01,

    weight_decay=0.0005,

    amp=True,

    cache=False,

    # Keep token/plate geometry.
    mosaic=0.0,
    mixup=0.0,

    degrees=0.0,
    translate=0.0,
    shear=0.0,
    perspective=0.0,

    scale=0.10,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.0,
    hsv_s=0.08,
    hsv_v=0.15,

    patience=10,

    project=str(
        RUN_ROOT
    ),

    name="bangla_lpdb_yolo11m_motion_finetune",

    exist_ok=True,

    save=True,

    save_period=5,

    plots=True,

    verbose=True
)

print("=" * 70)
print("FINE-TUNING FINISHED")
print("=" * 70)




STARTING FINE-TUNING
Starting model: /kaggle/input/notebooks/tanvirmahtabtaneem/ocr-model/bangla_lpdb_runs/bangla_lpdb_yolo11m_t4x2/weights/best.pt
Dataset: /kaggle/working/bangla_lpdb_finetune/data.yaml
GPU: 0
Image size: 960
Batch size: 4
Epochs: 35
New https://pypi.org/project/ultralytics/8.4.158 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/bangla_lpdb_finetune/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=35, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=

In [19]:
# ============================================================
# CELL 18 — VALIDATE THE NEW BEST MODEL
# ============================================================

best_path = (
    RUN_ROOT
    / "bangla_lpdb_yolo11m_motion_finetune"
    / "weights"
    / "best.pt"
)

print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Best model:")
print(best_path)

print(
    "Exists:",
    best_path.exists()
)

if not best_path.exists():
    raise FileNotFoundError(
        f"Fine-tuned best.pt not found:\n{best_path}"
    )

best_model = YOLO(
    str(best_path)
)

metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS
)

print("Validation results:")

print(
    "Precision:",
    metrics.box.mp
)

print(
    "Recall:",
    metrics.box.mr
)

print(
    "mAP50:",
    metrics.box.map50
)

print(
    "mAP50-95:",
    metrics.box.map
)



FINAL VALIDATION
Best model:
/kaggle/working/bangla_lpdb_finetune_runs/bangla_lpdb_yolo11m_motion_finetune/weights/best.pt
Exists: True
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,108,674 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1813.8±1142.7 MB/s, size: 101.2 KB)
val: Scanning /kaggle/working/bangla_lpdb_finetune/labels/val.cache... 266 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 266/266 111.6Mit/s 0.0s
val: /kaggle/working/bangla_lpdb_finetune/images/val/331.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 5.0it/s 13.4s
                   all        266       2230      0.932      0.935       0.97      0.708
                     0        123        154       0.98      0.968      0.991      0.667
                     1        222        378      0.989      0.

In [20]:
# ============================================================
# CELL 19 — SHOW OUTPUT LOCATION
# ============================================================

print("=" * 70)
print("DONE")
print("=" * 70)

print(
    "New best.pt:",
    best_path
)

print(
    "New model exists:",
    best_path.exists()
)

print(
    "\nDo not retrain the old model again."
)
print(
    "This best.pt is the fine-tuned model."
)




DONE
New best.pt: /kaggle/working/bangla_lpdb_finetune_runs/bangla_lpdb_yolo11m_motion_finetune/weights/best.pt
New model exists: True

Do not retrain the old model again.
This best.pt is the fine-tuned model.
